In [1]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from dotenv import load_dotenv
import os
from crewai import Agent, Task, Crew, Process, LLM
import os
from langchain_openai import ChatOpenAI
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv
# from langchain_groq import ChatGroq
import sys
import os
from pathlib import Path
from typing import Optional
import json
import argparse
import datetime
import time
import os
from crewai import LLM
from langchain_openai import ChatOpenAI

from crew_compliance_checker import ComplianceCheckerCrew
from crew_result_interpreter import ResultInterpreterCrew
from crew_sql_generation import SQLGenerationCrew

from langfuse import Langfuse, get_client
from openinference.instrumentation.crewai import CrewAIInstrumentor
from openinference.instrumentation.litellm import LiteLLMInstrumentor
from scoring import add_score_to_result
from test_utils import list_test_case_folders, get_test_case_by_name, run_mcp_server, update_env_db_path, load_questions_from_json, save_execution_info
from utils_db_helper import set_database_path, clean_sql_query, run_query

load_dotenv()

# Set Langfuse
langfuse = Langfuse(
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "http://localhost:3000"),
)

CrewAIInstrumentor().instrument(skip_dep_check=True)
LiteLLMInstrumentor().instrument()

# Model IDS
model_ids = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "qwen/qwen3-32b",
]

model_id_tuples = [
    (model_ids[0], model_ids[1], model_ids[2]),
    (model_ids[1], model_ids[2], model_ids[3]),
    (model_ids[2], model_ids[3], model_ids[1]),
    (model_ids[3], model_ids[1], model_ids[2]),
]

# Cases
cases = list_test_case_folders()
cases = sorted(cases, key=lambda x: x["name"])

# Utils
mock = False
if mock:
    from mock_crew_utils import run_sql_generation, post_sql_generation, run_compliance_checker, post_compliance_checker, run_result_interpretation, post_result_interpretation
else:
    from crew_utils import run_sql_generation, post_sql_generation, run_compliance_checker, post_compliance_checker, run_result_interpretation, post_result_interpretation

/home/ridwanfatur/portfolio/sql-assistant-mcp/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


In [3]:
def get_llm(model_id):
    return ChatOpenAI(
        openai_api_base="https://api.groq.com/openai/v1",
        openai_api_key=os.environ.get("GROQ_API_KEY"),
        temperature=0,
        model_name=f"groq/{model_id}",
        top_p=1,
        max_retries=3,
        request_timeout=60,
    )

### Planning

In [ ]:
for index_case in range(len(cases)):
    for index_model_tuple in range(len(model_id_tuples)):
        print(f"Case {index_case}, Model: {index_model_tuple}")
        ## Execute
        process(index_case, index_model_tuple)

Case 0, Model: 0
[Langfuse] Trace created with ID: dd84fee451e45f6db7e6b21022feb210
[Orchestrator] Execution attempt 1/3
[Business Logic] Attempt 1/3


/home/ridwanfatur/portfolio/sql-assistant-mcp/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'items', 'anyOf', 'enum', 'properties'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c39fc7af-c87a-4d54-9d60-5b1498d937ec                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert SQL Query Generator                                                                              │
│                                                                                                                 │
│  Task: MISSION: Generate a SQL query that answers this user question: "Name the score for toronto visitor and   │
│  record of 29-17-8"                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  STEP-BY-STEP PROCESS:                                                                                          │
│  1. Use get_database_schema tool to see all available tables, columns, and sample data                          │
│  2. Identify which tables and columns are needed to answer the question                                         │
│  3. If you need to understand data patterns, use get_table_sample or get_column_stats                           │
│  4. Write a SQL query using ONLY the tables and columns that exist in the schema                                │
│  5. Double-check that every table name and column name in your query exists in the schema                       │
│                                                                                                                 │
│  CRITICAL RULES:                                                                                                │
│  - NEVER invent or assume table/column names - use get_database_schema first                                    │
│  - Return ONLY the SQL query as plain text (no markdown, no ```sql, no explanations)                            │
│  - The query must be syntactically correct and executable                                                       │
│  - Use proper SQL syntax: JOINs, WHERE clauses, GROUP BY, ORDER BY, LIMIT as needed                             │
│  - If the question cannot be answered with available data, return: -- Cannot answer: [reason]                   │
│                                                                                                                 │
│  Database schema is available via tools. USER QUESTION: Name the score for toronto visitor and record of        │
│  29-17-8                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/ridwanfatur/portfolio/sql-assistant-mcp/.venv/lib/python3.12/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert SQL Query Generator                                                                              │
│                                                                                                                 │
│  Thought: Thought: I need to start by examining the database schema to understand available tables and columns  │
│                                                                                                                 │
│  Using Tool: get_database_schema                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"properties\": {}, \"title\": \"get_database_schemaArguments\", \"type\": \"object\"}"                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  DATABASE SCHEMA:                                                                                               │
│  ================================================================================                               │
│                                                                                                                 │
│                                                                                                                 │
│  Table: table_name_89                                                                                           │
│  --------------------------------------------------------------------------------                               │
│  Columns:                                                                                                       │
│    - score: TEXT                                                                                                │
│    - visitor: TEXT                                                                                              │
│    - record: TEXT                                                                                               │
│                                                                                                                 │
│  Row count: 16                                                                                                  │
│                                                                                                                 │
│  Sample data (first 3 rows):                                                                                    │
│  score  visitor   record                                                                                        │
│    4-3  toronto  35-15-6                                                                                        │
│    4-2  toronto  18-25-7                                                                                        │
│    6-3 montreal 23-19-10                                                                                        │
│                                                                                                                 │
│  ================================================================================                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 
[2026-05-14 19:16:15][INFO]: Max RPM reached, waiting for next minute to start.


### Main Function

In [21]:
def process(index_case, index_model_tuple):
    ## Set Configs
    # index_model_tuple = 1
    # index_case = 0
    max_crew_rpm = 1
    max_crew_retries = 2
    max_business_retries = 2 
    
    # Result
    workflow_result = None
    raw_sql = ""
    reviewed_sql = ""
    compliance_report = ""
    query_result = ""
    business_interpretation = "" 
    log_messages = []
    
    # Variable
    retry_count = 0
    trace_id = None

    ## Pre Evaluation
    # Variable by Config
    case = cases[index_case]
    test_case_name = case['name']
    test_case = get_test_case_by_name(test_case_name)
    db_path = test_case["database"]
    questions_file = test_case["questions"]
    question_data = load_questions_from_json(questions_file, 0)
    user_input = question_data.get("question", str(question_data))
    question_meta = {
        "difficulty": question_data.get("difficulty", "Unknown"),
        "category": question_data.get("category", "Unknown"),
    }
    
    # Update Env
    set_database_path(db_path)
    update_env_db_path(db_path) 
    
    # Variable by Config
    interpretation_result = None
    sql_generation_llm = get_llm(model_id_tuples[index_model_tuple][0])
    compliance_checker_llm = get_llm(model_id_tuples[index_model_tuple][1])
    result_interpreter_llm = get_llm(model_id_tuples[index_model_tuple][2])
    sql_generation_crew = SQLGenerationCrew(llm=sql_generation_llm, max_crew_rpm=max_crew_rpm)
    compliance_checker_crew = ComplianceCheckerCrew(llm=compliance_checker_llm, max_crew_rpm=max_crew_rpm)
    result_interpreter_crew = ResultInterpreterCrew(llm=result_interpreter_llm, max_crew_rpm=max_crew_rpm)  

    ### START TIME
    start_time = datetime.datetime.now()
    
    # Log
    log_messages.append(f"Using database: {db_path}")
    log_messages.append(f"User input: {user_input}")
    
    try:
        # Langfuse
        with langfuse.start_as_current_span(name="sql_assistant_workflow") as span:
            
            # Variable
            trace_id = span.trace_id
    
            # Print
            print(f"[Langfuse] Trace created with ID: {trace_id}")
            
            # Span
            span.update(metadata={
                "user_input": user_input[:100],
                "max_crew_retries": max_crew_retries,
            })
            
            # Loop
            while retry_count <= max_crew_retries and workflow_result is None:
                try:
                    # Variable
                    orchestrator_suffix = f" (retry {retry_count})" if retry_count > 0 else ""
    
                    # Langfuse
                    with langfuse.start_as_current_span(name=f"crew_execution{orchestrator_suffix}") as execution_span:
                        # Print
                        print(f"[Orchestrator] Execution attempt {retry_count + 1}/{max_crew_retries + 1}")
                        
                        # Span
                        execution_span.update(metadata={
                            "orchestrator_attempt": retry_count + 1,
                            "is_orchestrator_retry": retry_count > 0
                        })
                        
                        # Variable
                        business_retry_count = 0
                        previous_attempts = []
                        compliance_passed = False
    
                        # Loop
                        while business_retry_count <= max_business_retries:
                            
                            # Variable
                            attempt_num = business_retry_count + 1
                            business_suffix = f" (business_retry {business_retry_count})" if business_retry_count > 0 else ""
    
                            # Print
                            print(f"[Business Logic] Attempt {attempt_num}/{max_business_retries + 1}")
                            
                            # Langfuse
                            with langfuse.start_as_current_span(name=f"generate_sql{business_suffix}") as sql_span:
                                try:
                                    # Log
                                    log_messages.append(f"Step 1-2 (Attempt {attempt_num}): Generating and reviewing SQL query...")
                                    if previous_attempts:
                                        log_messages.append(f"Including context from {len(previous_attempts)} previous attempt(s)")
                                    
                                    # Crew
                                    generation_result = run_sql_generation(sql_generation_crew, user_input, previous_attempts)
                                    reviewed_sql = post_sql_generation(generation_result)
                                    
                                    # Log
                                    log_messages.append(f"Generated & Reviewed SQL: {reviewed_sql}")
                                    
                                    # Span
                                    sql_span.update(metadata={
                                        "sql_query": reviewed_sql[:200],
                                        "has_context": len(previous_attempts) > 0,
                                        "business_attempt": attempt_num,
                                        "completed": True
                                    })
                                except Exception as span_error:
                                    # Span
                                    sql_span.update(metadata={
                                        "error": str(span_error),
                                        "error_type": type(span_error).__name__,
                                        "completed": False
                                    })
                                    
                                    # Print
                                    print(f"[Langfuse] SQL generation span error: {span_error}")
                                    raise
    
                            # Langfuse
                            with langfuse.start_as_current_span(name=f"check_compliance{business_suffix}") as compliance_span:
                                try:
                                    # Log
                                    log_messages.append("Step 2: Compliance check...")
    
                                    # Crew
                                    compliance_result = run_compliance_checker(compliance_checker_crew, reviewed_sql)
                                    compliance_report = post_compliance_checker(compliance_result)
    
                                    # Log
                                    log_messages.append(f"Compliance report: {compliance_report}")
                                    
                                    # Variable
                                    compliance_passed = "verdict: pass" in compliance_report.lower()
                                    
                                    # Span
                                    compliance_span.update(metadata={
                                        "approved": compliance_passed,
                                        "business_attempt": attempt_num,
                                        "completed": True
                                    })
                                except Exception as span_error:
                                    # Span
                                    compliance_span.update(metadata={
                                        "error": str(span_error),
                                        "error_type": type(span_error).__name__,
                                        "completed": False
                                    })
                                    
                                    # Print
                                    print(f"[Langfuse] Compliance check span error: {span_error}")
                                    raise
                            
                            if compliance_passed:
                                # Print
                                print(f"[Business Logic] Compliance passed on attempt {attempt_num}")
                                break 
                            elif business_retry_count < max_business_retries:
                                # Span
                                previous_attempts.append({
                                    "sql": reviewed_sql,
                                    "compliance_report": compliance_report,
                                    "attempt_number": attempt_num
                                })
                                
                                # Log
                                log_messages.append(f"⚠️  Compliance check failed on attempt {attempt_num}. Will retry with context...")
    
                                # Print
                                print(f"[Business Logic] Compliance failed. Retry {business_retry_count + 1}/{max_business_retries}")
    
                                # Variable
                                business_retry_count += 1
                            else:
                                # Print
                                print(f"[Business Logic] Max attempts ({max_business_retries + 1}) reached")
                                break
    
                        # Log
                        log_messages.append("Step 3: Executing SQL query...")
                        
                        if compliance_passed:
                            # Variable
                            query_result = run_query(reviewed_sql, db_path)
    
                            # Log
                            log_messages.append(f"Query result:\n{query_result}")
    
                            # Languse
                            with langfuse.start_as_current_span(name="interpret_results") as interpret_span:
                                try:
                                    # Log
                                    log_messages.append("Step 4: Interpreting query results...")
    
                                    # Crew
                                    interpretation_result = run_result_interpretation(
                                        result_interpreter_crew, 
                                        user_input, 
                                        reviewed_sql, 
                                        query_result
                                    )
                                    business_interpretation = post_result_interpretation(interpretation_result) 
                                    
                                    # Log
                                    log_messages.append(
                                        f"Business interpretation:\n{business_interpretation}"
                                    )
                                    
                                    # Span
                                    interpret_span.update(metadata={
                                        "interpretation_length": len(business_interpretation),
                                        "completed": True
                                    })
                                except Exception as span_error:
                                    # Span
                                    interpret_span.update(metadata={
                                        "error": str(span_error),
                                        "error_type": type(span_error).__name__,
                                        "completed": False
                                    })
    
                                    # Print
                                    print(f"[Langfuse] Result interpretation span error: {span_error}")
                                    raise
                        else:
                            # Log
                            log_messages.append(
                                f"❌ SQL query failed compliance check after {business_retry_count + 1} attempts."
                            )
    
                            # Variable
                            query_result = f"Query not executed due to compliance issues after {business_retry_count + 1} attempts"
                            business_interpretation = "Interpretation not available, query was not executed"
                        
                        # Span
                        execution_span.update(metadata={
                            "business_retry_count": business_retry_count,
                            "total_business_attempts": business_retry_count + 1,
                            "compliance_passed": compliance_passed,
                            "compliance_failures": len(previous_attempts)
                        })
    
                        # Variable
                        workflow_result = True
    
                        # Print
                        print(f"[Orchestrator] Execution attempt {retry_count + 1} completed")
                except Exception as crew_error:
                    # Variable
                    retry_count += 1
                    error_msg = str(crew_error)
                    
                    # Print
                    print(f"[Orchestrator] Execution attempt {retry_count} failed with exception: {error_msg}")
                    
                    # Log
                    log_messages.append(f"Orchestrator attempt {retry_count} failed: {error_msg}")
                    
                    if retry_count <= max_crew_retries:
                        # Print
                        print(f"[Orchestrator] Retrying... (attempt {retry_count + 1}/{max_crew_retries + 1})")
                        
                        # Variable
                        raw_sql = ""
                        reviewed_sql = ""
                        compliance_report = ""
                        query_result = ""
                        business_interpretation = ""
                    else:
                        # Print
                        print(f"[Orchestrator] Max retries ({max_crew_retries}) reached. Giving up.")
                        raise crew_error
            
            # Span
            span.update(metadata={
                "completed": True,
                "orchestrator_retry_count": retry_count,
                "total_orchestrator_attempts": retry_count + 1,
            })
    
    except Exception as span_error:
        # Print
        print(f"[Langfuse] Failed to create span: {span_error}")
        raise
    
    try:
        langfuse.flush()
        
        # Print
        print("[Langfuse] Trace data flushed successfully")
    except Exception as flush_error:
        # Print
        print(f"[Langfuse] Failed to flush trace: {flush_error}")
    
    ### END TIME
    end_time = datetime.datetime.now()
    
    # Save Result
    result_data = {
        "user_input": user_input,
        "raw_sql": raw_sql,
        "reviewed_sql": reviewed_sql,
        "compliance_report": compliance_report,
        "query_result": query_result,
        "business_interpretation": business_interpretation,
        "database_path": db_path,
        "execution_time": end_time.isoformat(),
        "trace_id": trace_id if trace_id else "No trace ID available",
        "orchestrator_retry_count": retry_count,
        "total_orchestrator_attempts": retry_count + 1,
        "retry_info": "No orchestrator retries needed" if retry_count == 0 else f"Succeeded after {retry_count} orchestrator retry(s)",
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
        "duration_seconds": (end_time - start_time).total_seconds(),
    }
    result_data["difficulty"] = question_meta.get("difficulty", "Unknown")
    result_data["category"] = question_meta.get("category", "Unknown")
    result_data["token_usage_sql_generation"] = generation_result.token_usage.__dict__
    result_data["token_usage_compliance_checker"] = compliance_result.token_usage.__dict__
    result_data["token_usage_result_interpretation"] = interpretation_result.token_usage.__dict__ if interpretation_result is not None else {}
    
    result_data["model_sql_generation"] = model_id_tuples[index_model_tuple][0]
    result_data["model_compliance_checker"] = model_id_tuples[index_model_tuple][1]
    result_data["model_result_interpretation"] =  model_id_tuples[index_model_tuple][2]
    result_data["case"] =  index_case
    
    result_data = add_score_to_result(result_data)
    
    file_name = datetime.datetime.now().strftime("%d-%m-%Y %H-%M")
    result_file = Path("result/" + f"{file_name}.json")
    result_file.parent.mkdir(parents=True, exist_ok=True)
    with result_file.open("w", encoding="utf-8") as f:
        json.dump(result_data, f, ensure_ascii=False, indent=2)
        
    log_content = "\n".join(log_messages)
    log_file = Path("result/" + f"{file_name}.md")
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with open(log_file, "w", encoding="utf-8") as f:
        f.write(log_content)